# Browser History Domain Overview

This notebook gives a first rough overview of your browser history by grouping visits per domain.


In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path("BrowserHistory_22.02.26.csv")

# Load CSV with pandas, parse DateTime as index
df = pd.read_csv(
    csv_path,
    encoding="utf-8-sig",
    parse_dates=["DateTime"],
    index_col="DateTime"
)

print(f"Loaded {len(df):,} history rows from {csv_path.name}")
print(f"\nShape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nIndex range: {df.index.min()} to {df.index.max()}")

## 2) Extract domain from each URL
Normalizes domains by lowering case and stripping a leading `www.`.

In [ ]:
from urllib.parse import urlparse
from collections import Counter

def extract_domain(url: str) -> str:
    if not url:
        return "(empty)"

    parsed = urlparse(url)
    domain = parsed.netloc.lower().strip()

    if domain.startswith("www."):
        domain = domain[4:]

    return domain or "(no-domain)"

# Apply domain extraction to the NavigatedToUrl column
df["domain"] = df["NavigatedToUrl"].apply(extract_domain)

# Count domains
domain_counts = df["domain"].value_counts()

print(f"Unique domains: {len(domain_counts):,}")
print(f"Total visits: {domain_counts.sum():,}")

## 3) Top domains
Shows the highest-frequency domains for a rough overview.

In [ ]:
top_n = 25
top_domains = domain_counts.head(top_n)

print(f"Top {top_n} domains by visit count:\n")
for idx, (domain, count) in enumerate(top_domains.items(), start=1):
    print(f"{idx:>2}. {domain:<35} {count:>8,}")

## 4) Optional quick buckets
Creates coarse buckets to reduce noise from many one-off domains.

In [ ]:
bucketed = pd.Series(index=[">=100", "25-99", "5-24", "1-4"], dtype=int).fillna(0)

for count in domain_counts.values:
    if count >= 100:
        bucket = ">=100"
    elif count >= 25:
        bucket = "25-99"
    elif count >= 5:
        bucket = "5-24"
    else:
        bucket = "1-4"
    bucketed[bucket] += 1

for label in [">=100", "25-99", "5-24", "1-4"]:
    print(f"Domains with {label:>5} visits: {bucketed[label]:,}")

In [ ]:
from urllib.parse import urlparse, parse_qs

def is_google_search(url: str) -> bool:
    """Check if URL is a Google search"""
    if not url:
        return False
    parsed = urlparse(url)
    return parsed.netloc.lower() in ("www.google.com", "google.com", "google.de", "google.co.uk")

def extract_search_query(url: str) -> str:
    """Extract search query from Google URL"""
    if not url:
        return None
    parsed = urlparse(url)
    params = parse_qs(parsed.query)
    q = params.get("q", [None])[0]
    return q.replace("+", " ") if q else None

def has_ai_overview(url: str) -> bool:
    """Check if search used Google AI Overview feature (udm=50)"""
    if not url:
        return False
    parsed = urlparse(url)
    params = parse_qs(parsed.query)
    udm = params.get("udm", [None])[0]
    return udm == "50"

# Apply detection functions to the DataFrame
df["is_google_search"] = df["NavigatedToUrl"].apply(is_google_search)
df["search_query"] = df["NavigatedToUrl"].apply(lambda url: extract_search_query(url) if is_google_search(url) else None)
df["has_ai_overview"] = df["NavigatedToUrl"].apply(has_ai_overview)

# Stats
total_google_searches = df["is_google_search"].sum()
ai_searches = df["has_ai_overview"].sum()

print(f"Total Google searches: {total_google_searches:,}")
print(f"AI-powered searches (udm=50): {ai_searches:,}")
if total_google_searches > 0:
    pct = (ai_searches / total_google_searches) * 100
    print(f"Percentage using AI Overview: {pct:.1f}%")

## 6) Top AI Search Queries
Shows the most frequent queries using AI Overview feature.

In [ ]:
ai_queries = df[df["has_ai_overview"]]["search_query"].value_counts().head(15)

print(f"Top 15 AI-assisted search queries:\n")
for idx, (query, count) in enumerate(ai_queries.items(), start=1):
    if query:
        print(f"{idx:>2}. {query:<60} {count:>4,}")

## 7) All Google searches by topic
Groups all Google search queries (both AI-powered and traditional) by topic category and counts them.

In [ ]:
def classify_search_topic(query: str) -> str:
    """Classify a search query into a topic category"""
    if not query:
        return "Other"
    
    query_lower = query.lower()
    
    # Define topic keywords
    topics = {
        "AI/ML": ["copilot", "agent", "gpt", "llm", "neural", "ml", "machine learning", 
                  "deep learning", "ai", "chatgpt", "ort", "whisper", "mcp"],
        "Programming": ["python", "javascript", "react", "code", "debug", "node", "typescript", 
                        "function", "error", "api", "sdk", "library", "nodejs", "npm", "github"],
        "Hardware/Electronics": ["arduino", "raspberry", "circuit", "usb", "microcontroller", 
                                 "fnirsi", "multimeter", "instek", "breadboard", "sensor"],
        "Automotive/Vehicles": ["mercedes", "volvo", "car", "bmw", "automobile", "engine", 
                                "motor", "vehicle", "driving"],
        "Shopping/Ecommerce": ["amazon", "ebay", "shop", "buy", "price", "product", "sale", 
                               "aliexpress", "store"],
        "News/Current": ["news", "breaking", "update", "announced", "release", "launched"],
        "Tools/Utilities": ["tool", "utility", "software", "app", "application", "extension"],
        "General Info": ["what", "how", "why", "explain", "definition", "meaning", "vs", "vs."],
    }
    
    # Check each topic's keywords
    for topic, keywords in topics.items():
        for keyword in keywords:
            if keyword in query_lower:
                return topic
    
    return "Other"

# Extract all Google searches (not just AI-powered)
all_google_searches = df[df["is_google_search"]]["search_query"].dropna()

# Apply topic classification
topics = all_google_searches.apply(classify_search_topic)

# Count by topic
topic_counts = topics.value_counts().sort_values(ascending=False)

print(f"Search topics (all {len(all_google_searches):,} Google searches):\n")
for topic, count in topic_counts.items():
    pct = (count / len(all_google_searches)) * 100
    print(f"{topic:<20} {count:>6,} ({pct:>5.1f}%)")

## 8) Top searches by topic
Shows the most frequent queries in each topic category.

In [ ]:
# Create a DataFrame with queries and topics for better grouping
query_df = pd.DataFrame({
    "query": all_google_searches.values,
    "topic": topics.values
})

# Show top 5 queries per topic (excluding "Other" for readability)
for topic in topic_counts.index:
    if topic == "Other":
        continue  # Skip "Other" as it's too broad
    
    top_queries = query_df[query_df["topic"] == topic]["query"].value_counts().head(5)
    if len(top_queries) > 0:
        print(f"\n{topic}:")
        for query, count in top_queries.items():
            print(f"  • {query:<55} ({count})")

# Show sample of "Other" category
print(f"\nOther (sample of {topic_counts['Other']:,} queries):")
other_queries = query_df[query_df["topic"] == "Other"]["query"].value_counts().head(10)
for query, count in other_queries.items():
    print(f"  • {query:<55} ({count})")

## 9) Google visits without search queries
Breaks down the 617 Google visits that don't have an extractable search query.

In [ ]:
# Find Google visits that are NOT search queries
google_visits = df[df["is_google_search"]]
no_query_visits = google_visits[google_visits["search_query"].isna()]

print(f"Total Google.com visits:        {len(google_visits):,}")
print(f"Google searches (with query):   {len(all_google_searches):,}")
print(f"Google visits without query:    {len(no_query_visits):,}")
print(f"Difference:                     {len(google_visits) - len(all_google_searches):,}\n")

# Analyze non-search page titles to see what they were
print("Sample page titles from non-search Google visits:")
non_search_titles = no_query_visits["PageTitle"].dropna().value_counts().head(15)
for title, count in non_search_titles.items():
    print(f"  • {title:<60} ({count})")

## 10) Google search usage over time
Analyzes search query volume by day and week to show usage patterns and trends.

In [ ]:
# Get Google searches grouped by day
google_searches_daily = df[df["is_google_search"] & df["search_query"].notna()].copy()

# Group by date (day) and count queries
queries_per_day = google_searches_daily.groupby(google_searches_daily.index.date).size()

# Statistics
print(f"Google Search Usage Over Time")
print(f"{'='*50}")
print(f"Total days with searches: {len(queries_per_day):,}")
print(f"Date range: {queries_per_day.index.min()} to {queries_per_day.index.max()}")
print(f"\nDaily query statistics:")
print(f"  Average per day: {queries_per_day.mean():.1f}")
print(f"  Median per day:  {queries_per_day.median():.0f}")
print(f"  Max in a day:    {queries_per_day.max()}")
print(f"  Min in a day:    {queries_per_day.min()}")

# Show top 15 days
print(f"\nTop 15 days by search volume:")
top_days = queries_per_day.nlargest(15)
for i, (date, count) in enumerate(top_days.items(), start=1):
    print(f"{i:>2}. {date}  {count:>4} searches")

# Weekly aggregation to show trend
queries_per_week = google_searches_daily.resample("W").size()

print(f"\n\nWeekly search aggregation:")
print(f"{'='*50}")
print(f"Week Ending          Queries  Avg/Day")
print(f"{'-'*50}")
for week_end, count in queries_per_week.items():
    avg_per_day = count / 7
    print(f"{week_end.date()}     {count:>6}    {avg_per_day:>6.1f}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ------------------------------------------------------------------
# 11) All  visits over time
all_visits = df.copy()

# daily / weekly counts
visits_per_day = all_visits.groupby(all_visits.index.date).size()
visits_per_week = all_visits.resample("W").size()

print("Non‑Google visits over time")
print("="*50)
print(f"Days tracked: {len(visits_per_day):,}")
print(f"Date range: {visits_per_day.index.min()} to {visits_per_day.index.max()}")
print(f"\nDaily stats: avg {visits_per_day.mean():.1f}, max {visits_per_day.max()}, min {visits_per_day.min()}")

print("\nWeekly aggregation")
print("-"*50)
print("Week Ending          Visits  Avg/Day")
for week_end, cnt in visits_per_week.items():
    print(f"{week_end.date()}     {cnt:>6}    {cnt/7:>6.1f}")

# plot it with the same style as before
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

ax1.plot(visits_per_day.index, visits_per_day.values,
         linewidth=1.5, color="purple", marker="o", markersize=3, alpha=0.7)
ax1.set_title("Daily  Visits", fontsize=14, fontweight="bold")
ax1.set_ylabel("Visits", fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha="right")
ax1.axhline(visits_per_day.mean(), color="red", linestyle="--",
            linewidth=2, label=f"Average: {visits_per_day.mean():.1f}", alpha=0.7)
ax1.legend(fontsize=10)

ax2.bar(range(len(visits_per_week)), visits_per_week.values,
        width=0.6, color="olive", alpha=0.7, edgecolor="black")
ax2.set_title("Weekly  Visits", fontsize=14, fontweight="bold")
ax2.set_ylabel("Visits", fontsize=11)
ax2.set_xlabel("Week", fontsize=11)
ax2.grid(True, alpha=0.3, axis="y")
ax2.set_xticks(range(len(visits_per_week)))
ax2.set_xticklabels([d.strftime("%m-%d") for d in visits_per_week.index],
                    rotation=45, ha="right")
ax2.axhline(visits_per_week.mean(), color="red", linestyle="--",
            linewidth=2, label=f"Weekly Avg: {visits_per_week.mean():.0f}", alpha=0.7)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print(f"Daily rows: {len(visits_per_day)}  weeks: {len(visits_per_week)}")

## 12) Expanded Topic Classification – Phase 1
Classifies **all** visited URLs (not just Google searches) using a priority-based approach:
1. **Domain mapping** (`DOMAIN_CATEGORY_MAP`) — fastest, most reliable
2. **Page title keywords** — contextual clues
3. **Domain name keywords** — fallback for unmapped domains
4. **URL path keywords** — last resort

Bilingual keywords (English + German). First match wins.

In [ ]:
# ---------------------------------------------------------------------------
# Phase 1 – Domain → Category direct mapping (100+ common domains)
# ---------------------------------------------------------------------------
DOMAIN_CATEGORY_MAP = {
    # Programming
    "github.com": "Programming",
    "stackoverflow.com": "Programming",
    "gitlab.com": "Programming",
    "bitbucket.org": "Programming",
    "python.org": "Python",
    "pypi.org": "Python",
    "nodejs.org": "JavaScript/Web",
    "npmjs.com": "JavaScript/Web",
    "reactjs.org": "JavaScript/Web",
    "vuejs.org": "JavaScript/Web",
    "rust-lang.org": "Systems/Backend",
    "go.dev": "Systems/Backend",
    "docs.microsoft.com": "Programming",
    "learn.microsoft.com": "Programming",
    "developer.mozilla.org": "JavaScript/Web",
    "w3schools.com": "JavaScript/Web",
    "hub.docker.com": "Systems/Backend",
    "docker.com": "Systems/Backend",
    # AI / ML
    "openai.com": "AI/LLMs",
    "chat.openai.com": "AI/LLMs",
    "claude.ai": "AI/LLMs",
    "huggingface.co": "ML/Deep Learning",
    "tensorflow.org": "ML/Deep Learning",
    "pytorch.org": "ML/Deep Learning",
    "kaggle.com": "Data Science",
    "deepseek.com": "AI/LLMs",
    "chat.deepseek.com": "AI/LLMs",
    "perplexity.ai": "AI/LLMs",
    "copilot.microsoft.com": "AI/LLMs",
    # Entertainment
    "youtube.com": "Entertainment/Media",
    "netflix.com": "Entertainment/Media",
    "twitch.tv": "Entertainment/Media",
    "imdb.com": "Entertainment/Media",
    "spotify.com": "Entertainment/Media",
    "reddit.com": "Entertainment/Media",
    "tiktok.com": "Entertainment/Media",
    "instagram.com": "Entertainment/Media",
    "facebook.com": "Entertainment/Media",
    "twitter.com": "Entertainment/Media",
    "x.com": "Entertainment/Media",
    "primevideo.com": "Entertainment/Media",
    "disneyplus.com": "Entertainment/Media",
    "store.steampowered.com": "Entertainment/Media",
    "steampowered.com": "Entertainment/Media",
    # Shopping
    "amazon.com": "Shopping/Ecommerce",
    "amazon.de": "Shopping/Ecommerce",
    "ebay.com": "Shopping/Ecommerce",
    "ebay.de": "Shopping/Ecommerce",
    "aliexpress.com": "Shopping/Ecommerce",
    "etsy.com": "Shopping/Ecommerce",
    "otto.de": "Shopping/Ecommerce",
    "zalando.de": "Shopping/Ecommerce",
    "mediamarkt.de": "Shopping/Ecommerce",
    "idealo.de": "Shopping/Ecommerce",
    "geizhals.de": "Shopping/Ecommerce",
    "conrad.de": "Shopping/Ecommerce",
    # News
    "bbc.com": "News/Current",
    "cnn.com": "News/Current",
    "dw.com": "News/Current",
    "spiegel.de": "News/Current",
    "zeit.de": "News/Current",
    "heise.de": "News/Current",
    "golem.de": "News/Current",
    "ard.de": "News/Current",
    "tagesschau.de": "News/Current",
    "sueddeutsche.de": "News/Current",
    "faz.net": "News/Current",
    # Travel
    "booking.com": "Travel/Location",
    "airbnb.com": "Travel/Location",
    "expedia.com": "Travel/Location",
    "tripadvisor.com": "Travel/Location",
    "maps.google.com": "Travel/Location",
    "openstreetmap.org": "Travel/Location",
    "flightradar24.com": "Travel/Location",
    # Finance
    "coinbase.com": "Finance/Investment",
    "crypto.com": "Finance/Investment",
    "investing.com": "Finance/Investment",
    "coingecko.com": "Finance/Investment",
    "binance.com": "Finance/Investment",
    "tradingview.com": "Finance/Investment",
    "finanzen.net": "Finance/Investment",
    # Work
    "linkedin.com": "Work/Career",
    "indeed.com": "Work/Career",
    "xing.com": "Work/Career",
    # Health
    "webmd.com": "Health/Medical",
    "mayoclinic.org": "Health/Medical",
    "healthline.com": "Health/Medical",
    "netdoktor.de": "Health/Medical",
    # Education
    "udemy.com": "Education/Learning",
    "coursera.org": "Education/Learning",
    "edx.org": "Education/Learning",
    "khanacademy.org": "Education/Learning",
    "codecademy.com": "Education/Learning",
    "wikipedia.org": "General Info",
    # Tools / Utilities
    "notion.so": "Tools/Utilities",
    "trello.com": "Tools/Utilities",
    "figma.com": "Tools/Utilities",
    "canva.com": "Tools/Utilities",
    "obsidian.md": "Tools/Utilities",
}

# ---------------------------------------------------------------------------
# Expanded topic keyword lists – 21 categories, bilingual (EN + DE)
# ---------------------------------------------------------------------------
TOPICS = {
    # Programming sub-categories
    "Python": [
        "python", "django", "flask", "pip", "pandas", "numpy", "jupyter",
        "anaconda", "virtualenv", "pycharm", "pytest", "poetry", "fastapi",
        "sqlalchemy", "python-programmierung", "pythonista",
    ],
    "JavaScript/Web": [
        "javascript", "react", "nodejs", "node.js", "typescript", "html",
        "css", "vue", "angular", "npm", "webpack", "vite", "svelte",
        "next.js", "nuxt", "tailwind", "bootstrap", "frontend",
        "web development", "webentwicklung",
    ],
    "Systems/Backend": [
        "rust", "golang", "c++", "c#", "java", "kotlin", "backend",
        "microservice", "postgresql", "mysql", "redis", "docker",
        "kubernetes", "devops", "linux", "systemd", "datenbank",
    ],
    "Mobile Dev": [
        "swift", "ios", "android", "flutter", "react native",
        "xcode", "android studio", "app development", "mobilentwicklung",
    ],
    # AI / ML sub-categories
    "AI/LLMs": [
        "copilot", "agent", "gpt", "llm", "chatgpt", "openai", "claude",
        "deepseek", "prompt", "gemini", "mistral", "ollama", "rag",
        "langchain", "llama", "mcp", "whisper", "ki", "chatbot",
        "sprachmodell", "kuenstliche intelligenz",
    ],
    "Data Science": [
        "data science", "data analysis", "visualization", "tableau",
        "power bi", "seaborn", "plotly", "notebook", "statistics",
        "regression", "datenanalyse", "datenwissenschaft",
    ],
    "ML/Deep Learning": [
        "neural", "deep learning", "tensorflow", "pytorch", "keras",
        "scikit", "sklearn", "reinforcement", "transformer", "fine-tuning",
        "embedding", "maschinelles lernen", "neuronales netz",
    ],
    # Hardware & Engineering
    "Hardware/Electronics": [
        "arduino", "raspberry", "circuit", "usb", "microcontroller",
        "fnirsi", "multimeter", "instek", "breadboard", "sensor",
        "electronics", "pcb", "soldering", "oscilloscope", "embedded",
        "schematic", "platine", "loetkolben", "widerstand",
    ],
    "Automotive/Vehicles": [
        "mercedes", "volvo", "bmw", "audi", "volkswagen", "toyota",
        "automobile", "engine", "motor", "vehicle", "driving", "repair",
        "maintenance", "fuel", "tire", "tesla", "electric vehicle",
        "auto", "kfz", "fahrzeug", "reparatur", "wartung",
    ],
    # Lifestyle domains
    "Health/Medical": [
        "health", "medical", "disease", "symptom", "doctor", "treatment",
        "medication", "hospital", "therapy", "wellness", "exercise",
        "nutrition", "mental health", "anxiety", "depression", "fitness",
        "gesundheit", "arzt", "krankheit", "medikament", "behandlung",
        "ernaehrung",
    ],
    "Entertainment/Media": [
        "movie", "film", "show", "gaming", "stream",
        "actor", "imdb", "music", "song", "artist", "album", "concert",
        "anime", "manga", "podcast", "serie", "musik", "spiel",
    ],
    "Travel/Location": [
        "hotel", "flight", "airport", "booking", "vacation", "travel",
        "route", "directions", "tourism", "hostel", "resort", "ticket",
        "reise", "urlaub", "flug", "stadtplan",
    ],
    "Finance/Investment": [
        "stock", "crypto", "bitcoin", "finance", "investment", "bank",
        "loan", "mortgage", "insurance", "tax", "savings", "trading",
        "portfolio", "ethereum", "defi",
        "aktie", "finanzen", "investition", "steuer", "versicherung",
        "krypto", "ersparnisse",
    ],
    "Sports": [
        "league", "football", "basketball", "soccer", "tennis",
        "championship", "tournament", "bundesliga", "formula 1", "f1",
        "cycling", "marathon", "fussball", "spieler", "mannschaft", "liga",
    ],
    "Food/Cooking": [
        "recipe", "cook", "food", "restaurant", "ingredient", "cooking",
        "baking", "cuisine", "dish", "menu", "delivery", "cafe", "vegan",
        "rezept", "kochen", "essen", "zutaten", "backen", "gericht",
    ],
    "Work/Career": [
        "resume", "interview", "salary", "employment", "career",
        "business", "company", "hiring", "recruit", "freelance", "startup",
        "bewerbung", "gehalt", "unternehmen", "karriere",
    ],
    "Education/Learning": [
        "tutorial", "learn", "class", "school", "university",
        "degree", "textbook", "exam", "study", "certification",
        "udemy", "coursera", "lecture",
        "kurs", "lernen", "schule", "studium", "pruefung", "zertifikat",
    ],
    # Broader categories (checked later in priority order)
    "Shopping/Ecommerce": [
        "amazon", "ebay", "shop", "buy", "price", "product", "sale",
        "aliexpress", "store", "deal", "coupon", "discount", "purchase",
        "checkout", "retail",
        "kaufen", "preis", "angebot", "rabatt", "einkaufen",
    ],
    "News/Current": [
        "news", "breaking", "update", "announced", "release", "launched",
        "latest", "announcement", "bulletin",
        "nachrichten", "aktuell", "meldung", "bericht",
    ],
    "Tools/Utilities": [
        "tool", "utility", "software", "application", "extension",
        "download", "installer", "plugin", "addon", "script", "cli",
        "command line", "terminal", "editor", "ide",
        "werkzeug", "programm", "herunterladen",
    ],
    "General Info": [
        "what is", "how to", "why", "explain", "definition", "meaning",
        " vs ", "comparison", "guide", "understand",
        "was ist", "wie", "warum", "erklaerung", "bedeutung", "vergleich",
        "anleitung",
    ],
}

print(f"DOMAIN_CATEGORY_MAP: {len(DOMAIN_CATEGORY_MAP)} domains")
print(f"TOPICS: {len(TOPICS)} categories, "
      f"{sum(len(v) for v in TOPICS.values())} total keywords")

In [ ]:
def _match_keywords(text: str) -> str:
    """Return first matching TOPICS category, or 'Other'."""
    if not text:
        return "Other"
    text_lower = text.lower()
    for topic, keywords in TOPICS.items():
        for kw in keywords:
            if kw in text_lower:
                return topic
    return "Other"


def classify_visit(url: str, domain: str, page_title: str) -> str:
    """Classify a browser visit using priority-based matching.

    Priority:
    1. Domain mapping (DOMAIN_CATEGORY_MAP)
    2. Page title keywords
    3. Domain name keywords
    4. URL path keywords
    """
    url = url if isinstance(url, str) else ""
    domain = domain if isinstance(domain, str) else ""
    page_title = page_title if isinstance(page_title, str) else ""

    # 1. Direct domain lookup
    if domain in DOMAIN_CATEGORY_MAP:
        return DOMAIN_CATEGORY_MAP[domain]

    # 2. Page title
    result = _match_keywords(page_title)
    if result != "Other":
        return result

    # 3. Domain name itself
    result = _match_keywords(domain)
    if result != "Other":
        return result

    # 4. Full URL (last resort)
    return _match_keywords(url)


# Apply to ALL rows in the DataFrame
page_title_col = "PageTitle" if "PageTitle" in df.columns else None

df["category"] = df.apply(
    lambda row: classify_visit(
        row["NavigatedToUrl"] if pd.notna(row["NavigatedToUrl"]) else "",
        row["domain"] if pd.notna(row["domain"]) else "",
        row[page_title_col] if (page_title_col and pd.notna(row[page_title_col])) else "",
    ),
    axis=1,
)

print(f"Classified {len(df):,} total visits")

In [ ]:
# Category distribution across ALL visits
category_counts = df["category"].value_counts()
total = len(df)

print(f"Category distribution – all {total:,} visits\n")
print(f"{'Category':<25} {'Count':>8}  {'%':>6}")
print("-" * 45)
for cat, count in category_counts.items():
    pct = count / total * 100
    print(f"{cat:<25} {count:>8,}  {pct:>5.1f}%")

other_pct = category_counts.get("Other", 0) / total * 100
# Threshold from Phase 1 plan: investigate clustering if > 40% falls into "Other"
OTHER_THRESHOLD_PCT = 40
print(f"\n\u2192 'Other' coverage: {other_pct:.1f}% ",
      "(Phase 2 clustering recommended)" if other_pct > OTHER_THRESHOLD_PCT else "(coverage acceptable)")

# Top 5 domains per non-Other category
print("\n\nTop 5 domains per category (excluding 'Other'):\n")
for cat in category_counts.index:
    if cat == "Other":
        continue
    top_domains = df[df["category"] == cat]["domain"].value_counts().head(5)
    domain_str = ", ".join(f"{d}({c})" for d, c in top_domains.items())
    print(f"{cat:<25} {domain_str}")

## 13) Phase 2 – Unsupervised Clustering of 'Other' Visits
Uses **TF-IDF char n-grams** + **KMeans** to discover natural groupings inside the
unclassified 'Other' bucket, helping identify topics that the keyword lists missed.

In [ ]:
import warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")

other_df = df[df["category"] == "Other"].copy()
print(f"'Other' visits: {len(other_df):,} ({len(other_df) / len(df) * 100:.1f}% of total)")

# Minimum "Other" entries needed to make clustering statistically meaningful
MIN_CLUSTER_SIZE = 30
if len(other_df) < MIN_CLUSTER_SIZE:
    print("Not enough 'Other' entries to cluster meaningfully — skipping.")
else:
    page_title_col = "PageTitle" if "PageTitle" in other_df.columns else None
    title_series = (
        other_df[page_title_col].fillna("") if page_title_col else pd.Series("", index=other_df.index)
    )
    other_df["_cluster_text"] = other_df["domain"].fillna("") + " " + title_series

    # MAX_CLUSTERS caps granularity; MIN_CLUSTERS ensures meaningful groups;
    # CLUSTER_SIZE_RATIO: aim for ~50 visits per cluster
    MAX_CLUSTERS = 20
    MIN_CLUSTERS = 5
    CLUSTER_SIZE_RATIO = 50
    n_clusters = min(MAX_CLUSTERS, max(MIN_CLUSTERS, len(other_df) // CLUSTER_SIZE_RATIO))

    vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 3), max_features=500)
    X = vectorizer.fit_transform(other_df["_cluster_text"])

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    other_df["cluster"] = kmeans.fit_predict(X)

    print(f"\nDiscovered {n_clusters} clusters in 'Other' visits.")
    print("Review these to find new keyword categories:\n")
    print(f"{'Cluster':>8}  {'Size':>6}  Top domains (count)")
    print("-" * 70)
    for cid in range(n_clusters):
        rows = other_df[other_df["cluster"] == cid]
        top = rows["domain"].value_counts().head(4)
        top_str = ", ".join(f"{d}({c})" for d, c in top.items())
        print(f"{cid:>8}  {len(rows):>6}  {top_str}")

    # Expose cluster labels on the main df for further inspection
    # Use boolean-mask positional assignment to avoid duplicate-index reindex errors
    other_mask = df["category"] == "Other"
    df.loc[other_mask, "other_cluster"] = other_df["cluster"].astype("Int64").to_numpy()
    print("\nCluster IDs written to df['other_cluster'] for further inspection.")

In [ ]:
df['other_cluster'] 